---
tags: [algorithm, chemistry, resource-estimation, simulation]
---

# FTQC Chemistry Resource Estimation

This notebook shows how to compare fault-tolerant quantum chemistry resource
models with Qamomile. It focuses on algorithm-level quantities that matter
before a full logical circuit is available: Hamiltonian normalization,
phase-estimation iterations, Toffoli or T counts, logical qubits, physical
qubits, and runtime proxies.

In [1]:
# Install the latest Qamomile through pip!
# !pip install qamomile

In [2]:
import sympy as sp

from qamomile.circuit.estimator.algorithmic import (
    ChemistryQPEMethod,
    FTQCCostModel,
    estimate_qubitized_chemistry_qpe,
    estimate_single_ancilla_trotter_qpe,
)

## Background

Fault-tolerant chemistry algorithms are usually compared through different
resource quantities than NISQ variational examples. For qubitized QPE, the
Hamiltonian block-encoding normalization controls the number of walk calls,
while the per-walk implementation controls the Toffoli count. Recent
chemistry proposals try to reduce these costs by changing the Hamiltonian
representation rather than changing only backend-level gate decompositions.

Examples include symmetry-compressed double factorization
([arXiv:2403.03502](https://arxiv.org/abs/2403.03502)), simultaneous symmetry
shifts and tensor factorizations
([arXiv:2412.01338](https://arxiv.org/abs/2412.01338)), and early-FTQC
single-ancilla QPE with unitary weight concentration
([arXiv:2603.22778](https://arxiv.org/abs/2603.22778)). The estimators below
are deliberately symbolic. They let you test how a proposed representation
changes the cost-driving quantities before committing to a concrete
Hamiltonian-loading circuit.

## Problem Settings

We will compare three scenarios for the same active-space scale:

1. a tensor-hypercontraction-like qubitized QPE baseline,
2. a symmetry-compressed double-factorization-style qubitized QPE estimate,
3. an early-FTQC single-ancilla Trotter QPE estimate with a unitary-weight
   concentration factor.

The numbers are intentionally small and synthetic so the notebook is fast and
reviewable. They should be read as a workflow demonstration, not as a claim
about a particular molecule.

In [3]:
n_spin_orbitals = 40
precision = sp.Float("0.0016")  # about chemical accuracy in Hartree

cost_model = FTQCCostModel(
    physical_qubits_per_logical=800,
    logical_cycle_time_seconds=sp.Float("1e-6"),
    factory_qubits=20000,
    toffoli_throughput_per_second=sp.Float("2e6"),
)

## Qubitized QPE Comparison

Qamomile keeps the chemistry factorization cost external to the IR. The
estimator accepts the representation-dependent normalization and one-walk
Toffoli cost as inputs:

```text
qpe_iterations = lambda_norm / precision
toffoli_gates = qpe_iterations * walk_cost_toffoli
```

This keeps the model honest: reducing Hamiltonian normalization, reducing the
walk circuit, and changing physical architecture are separate design choices.

In [4]:
thc = estimate_qubitized_chemistry_qpe(
    n_spin_orbitals=n_spin_orbitals,
    lambda_norm=sp.Float("2.0e5"),
    precision=precision,
    walk_cost_toffoli=sp.Integer(4_000),
    method=ChemistryQPEMethod.TENSOR_HYPERCONTRACTION,
    cost_model=cost_model,
)

scdf = estimate_qubitized_chemistry_qpe(
    n_spin_orbitals=n_spin_orbitals,
    lambda_norm=sp.Float("1.0e5"),
    precision=precision,
    walk_cost_toffoli=sp.Integer(4_400),
    method=ChemistryQPEMethod.SYMMETRY_COMPRESSED_DF,
    second_factor_rank=9,
    cost_model=cost_model,
)

assert scdf.qpe_iterations < thc.qpe_iterations
assert scdf.toffoli_gates < thc.toffoli_gates

print("THC Toffoli gates:", sp.N(thc.toffoli_gates, 4))
print("SCDF-style Toffoli gates:", sp.N(scdf.toffoli_gates, 4))
print("SCDF-style logical qubits:", scdf.logical_qubits)

THC Toffoli gates: 5.000e+11
SCDF-style Toffoli gates: 2.750e+11
SCDF-style logical qubits: 120


## Early-FTQC Trotter QPE

Early-FTQC proposals may favor shallower single-ancilla QPE and Pauli
rotations over qubitized walks when qubits and depth are tightly constrained.
The unitary-weight concentration factor below represents a spectrally
invariant Hamiltonian transformation that reduces the cost-driving effective
weight.

In [5]:
plain_trotter = estimate_single_ancilla_trotter_qpe(
    n_spin_orbitals=n_spin_orbitals,
    n_pauli_terms=20_000,
    lambda_norm=sp.Float("2.0e5"),
    precision=precision,
    trotter_steps_per_sample=8,
    samples=128,
    cost_model=cost_model,
)

uwc_trotter = estimate_single_ancilla_trotter_qpe(
    n_spin_orbitals=n_spin_orbitals,
    n_pauli_terms=20_000,
    lambda_norm=sp.Float("2.0e5"),
    precision=precision,
    trotter_steps_per_sample=8,
    samples=128,
    unitary_weight_factor=sp.Float("0.1"),
    randomized_compilation_factor=sp.Float("0.5"),
    rotation_synthesis_t_gates=3,
    cost_model=cost_model,
)

assert uwc_trotter.qpe_iterations < plain_trotter.qpe_iterations
assert uwc_trotter.logical_depth < plain_trotter.logical_depth

print("Plain Trotter QPE depth proxy:", sp.N(plain_trotter.logical_depth, 4))
print("UWC-style Trotter QPE depth proxy:", sp.N(uwc_trotter.logical_depth, 4))
print("UWC-style T gates:", sp.N(uwc_trotter.t_gates, 4))

Plain Trotter QPE depth proxy: 2.560e+15
UWC-style Trotter QPE depth proxy: 1.280e+14
UWC-style T gates: 3.840e+14


## Result

We can put the estimates into a compact table. The important point is that
each column has a distinct design meaning: changing the Hamiltonian
representation should affect `qpe_iterations` and per-step cost, while
changing the hardware model should affect `physical_qubits` and runtime.

In [6]:
rows = [
    ("THC qubitized QPE", thc),
    ("SCDF-style qubitized QPE", scdf),
    ("Plain Trotter QPE", plain_trotter),
    ("UWC-style Trotter QPE", uwc_trotter),
]

for name, estimate in rows:
    print(
        name,
        {
            "logical_qubits": sp.N(estimate.logical_qubits, 4),
            "physical_qubits": sp.N(estimate.physical_qubits, 4),
            "qpe_iterations": sp.N(estimate.qpe_iterations, 4),
            "toffoli_gates": sp.N(estimate.toffoli_gates, 4),
            "t_gates": sp.N(estimate.t_gates, 4),
            "runtime_seconds": sp.N(estimate.runtime_seconds, 4),
        },
    )

assert scdf.physical_qubits > thc.physical_qubits
assert uwc_trotter.physical_qubits == plain_trotter.physical_qubits

THC qubitized QPE {'logical_qubits': 40.00, 'physical_qubits': 5.200e+4, 'qpe_iterations': 1.250e+8, 'toffoli_gates': 5.000e+11, 't_gates': 0, 'runtime_seconds': 5.000e+5}
SCDF-style qubitized QPE {'logical_qubits': 120.0, 'physical_qubits': 1.160e+5, 'qpe_iterations': 6.250e+7, 'toffoli_gates': 2.750e+11, 't_gates': 0, 'runtime_seconds': 2.750e+5}
Plain Trotter QPE {'logical_qubits': 41.00, 'physical_qubits': 5.280e+4, 'qpe_iterations': 1.250e+8, 'toffoli_gates': 0, 't_gates': 2.560e+15, 'runtime_seconds': 2.560e+9}
UWC-style Trotter QPE {'logical_qubits': 41.00, 'physical_qubits': 5.280e+4, 'qpe_iterations': 1.250e+7, 'toffoli_gates': 0, 't_gates': 3.840e+14, 'runtime_seconds': 1.920e+8}


## Summary

In this notebook, we:

- Separated FTQC chemistry resource quantities into Hamiltonian
  normalization, QPE iterations, non-Clifford counts, logical qubits,
  physical qubits, and runtime proxies.
- Compared qubitized QPE representations without lowering the Qamomile IR
  into backend-specific chemistry loading circuits.
- Demonstrated how a unitary-weight concentration factor can be modeled as a
  cost-driver reduction for early-FTQC single-ancilla Trotter QPE.